# Homework 11: Tendons, Speed, and Cocontraction 
**Computational Sensorimotor Control**

Two problems exploring how tendon compliance and cocontraction interact with EPH control.

**Problem 1 (Speed–Accuracy Tradeoff):** Sweep ramp duration across 400, 600, 800 ms and compare overshoot, peak velocity, and settling across all three muscle models.

**Problem 2 (Cocontraction and the Tendon):** Vary the C-command and discover that cocontraction shifts the equilibrium position in Hill models but not in the Gribble model.

---
## Setup

In [ ]:
!pip install --break-system-packages -q git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    'figure.figsize': (10, 5), 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.family': 'serif'
})

from smc import (
    Arm, Q_REF, make_muscles, make_hill_muscles,
    lambda_for_posture, make_ramp,
    simulate_lambda, simulate_hill, simulate_kmhm,
)

arm = Arm()
fk = arm.forward_kinematics
ik = arm.inverse_kinematics
start_hand = np.array(fk(Q_REF))
dt = 0.0001
T_SIM = 2.0

target_xy = start_hand + np.array([0.12, 0.0])
q_target = ik(target_xy[0], target_xy[1])

# Colors
C_CA   = '#2E86AB'
C_HILL = '#E74C3C'
C_KMHM = '#8E44AD'

print(f"Start: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm")
print(f"Target: ({target_xy[0]*100:.1f}, {target_xy[1]*100:.1f}) cm")

---
## Problem 1: Speed–Accuracy Tradeoff

In the lab, you compared a slow reach (500 ms ramp) and a fast reach (150 ms ramp).
The SE catapult produced 13% overshoot at fast speed but only 2% at slow speed.
Where is the transition? Sweep ramp durations of **400, 600, and 800 ms** for all
three models and characterize the speed–accuracy tradeoff.

### Part A: Run the sweep

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Sweep ramp durations and collect metrics for each model
# ════════════════════════════════════════════════════════════════
ramp_durations = [0.40, 0.60, 0.80]

def compute_metrics(t, h):
    v = np.linalg.norm(np.diff(h, axis=0) / dt, axis=1) * 100
    d = np.linalg.norm(h - start_hand, axis=1) * 100
    fd = d[-1]
    pv = np.max(v)
    ovsh = (np.max(d) - fd) / fd * 100 if fd > 0 else 0
    err = np.linalg.norm(h[-1] - target_xy) * 100
    # Movement time: 5% to 95%
    t5 = t95 = 0
    for k in range(len(d)):
        if t5 == 0 and d[k] >= 0.05 * fd: t5 = t[k]
        if t95 == 0 and d[k] >= 0.95 * fd: t95 = t[k]; break
    tmov = (t95 - t5) * 1000
    return {'pv': pv, 'ovsh': ovsh, 'err': err, 'tmov': tmov}

results = {name: [] for name in ['R-C Ca²⁺ λ', 'R-C Hill', 'KMHM']}

for dur in ramp_durations:
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    lam_fn = make_ramp(li, lf, t_start=0.05, duration=dur)

    t1, _, h1, _ = simulate_lambda(lam_fn, T=T_SIM)
    t2, _, h2, _ = simulate_hill(lam_fn, T=T_SIM)
    t3, _, h3, _ = simulate_kmhm(lam_fn, q_target, T=T_SIM, ramp_dur=dur)

    results['R-C Ca²⁺ λ'].append(compute_metrics(t1, h1))
    results['R-C Hill'].append(compute_metrics(t2, h2))
    results['KMHM'].append(compute_metrics(t3, h3))
    print(f"Ramp {dur*1000:.0f} ms done")
# ════════════════════════════════════════════════════════════════

### Part B: Speed–accuracy curves

Plot peak velocity vs overshoot for each model across the three ramp durations.

In [ ]:
# ── Figure 1: Speed–accuracy tradeoff ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

markers = {'R-C Ca²⁺ λ': 'o', 'R-C Hill': 's', 'KMHM': '^'}
colors = {'R-C Ca²⁺ λ': C_CA, 'R-C Hill': C_HILL, 'KMHM': C_KMHM}

ax = axes[0]; ax.set_title('A: Peak velocity vs overshoot', fontweight='bold')
for name in results:
    pvs = [m['pv'] for m in results[name]]
    ovshs = [m['ovsh'] for m in results[name]]
    ax.plot(pvs, ovshs, marker=markers[name], color=colors[name], lw=2,
            ms=10, label=name)
    for i, dur in enumerate(ramp_durations):
        ax.annotate(f'{dur*1000:.0f}ms', (pvs[i]+0.5, ovshs[i]+0.2), fontsize=8)
ax.set_xlabel('Peak velocity (cm/s)'); ax.set_ylabel('Overshoot (%)')
ax.legend(fontsize=9)

ax = axes[1]; ax.set_title('B: Movement time vs overshoot', fontweight='bold')
for name in results:
    tmovs = [m['tmov'] for m in results[name]]
    ovshs = [m['ovsh'] for m in results[name]]
    ax.plot(tmovs, ovshs, marker=markers[name], color=colors[name], lw=2,
            ms=10, label=name)
    for i, dur in enumerate(ramp_durations):
        ax.annotate(f'{dur*1000:.0f}ms', (tmovs[i]+10, ovshs[i]+0.2), fontsize=8)
ax.set_xlabel('Movement time (ms)'); ax.set_ylabel('Overshoot (%)')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

### Part C: Displacement traces at three speeds

Show displacement vs time for all three models at each ramp duration.

In [ ]:
# ── Figure 2: Displacement traces at 3 speeds ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, dur in enumerate(ramp_durations):
    ax = axes[idx]
    ax.set_title(f'Ramp = {dur*1000:.0f} ms', fontweight='bold')
    lam_fn = make_ramp(li, lf, t_start=0.05, duration=dur)

    for sim_fn, name, c in [(simulate_lambda, 'R-C Ca²⁺ λ', C_CA),
                             (simulate_hill, 'R-C Hill', C_HILL)]:
        t, _, h, _ = sim_fn(lam_fn, T=T_SIM)
        ax.plot(t*1000, np.linalg.norm(h - start_hand, axis=1)*100,
                color=c, lw=2, label=name)

    t, _, h, _ = simulate_kmhm(lam_fn, q_target, T=T_SIM, ramp_dur=dur)
    ax.plot(t*1000, np.linalg.norm(h - start_hand, axis=1)*100,
            color=C_KMHM, lw=2, label='KMHM')

    ax.axhline(12, color='gray', ls=':', lw=1)
    ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
    ax.legend(fontsize=7); ax.set_xlim(0, 2000)

plt.tight_layout(); plt.show()

### Part D: Questions

**Q1.** At which ramp duration does R-C Hill's overshoot first exceed 5%? Does KMHM shift this threshold?

**Q2.** The Gribble model (R-C Ca²⁺ λ) is slower but more accurate at every speed. Is this a feature or a limitation? What does this tell us about why real muscles have tendons despite the stability cost?

**Q3.** If you were designing a prosthetic arm with an elastic tendon, which controller (R-C Hill or KMHM) would you choose, and why?

*Write 1–2 sentences per question.*

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Your answers (as comments or print statements)
# ════════════════════════════════════════════════════════════════

# Q1:
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# shifts the threshold to a shorter ramp duration.

# Q2:
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# speed advantage outweighs the stability cost — and the nervous
# system uses feedback (Week 12: OFC) to manage the oscillations.

# Q3:
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# arm needs both speed and stability.
# ════════════════════════════════════════════════════════════════

---
## Problem 2: Cocontraction and the Tendon

In the Gribble model (Week 4), the C-command increases stiffness without
shifting the equilibrium position. Kistemaker et al. (2006) demonstrated that
with tendon compliance, this is no longer true — cocontraction shifts the
equilibrium because it changes the CE–SE force balance.

### Part A: Sweep cocontraction levels

Use the fast ramp (150 ms) and vary C from 0.10 to 0.75. Measure the final
hand position for each model.

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Sweep C values and record final hand position for each model
# ════════════════════════════════════════════════════════════════
C_values = [0.10, 0.25, 0.50, 0.75]
RAMP_FAST = 0.15

ep_results = {name: [] for name in ['R-C Ca²⁺ λ', 'R-C Hill', 'KMHM']}

for C in C_values:
    ...  # YOUR CODE HERE
    ...  # YOUR CODE HERE
    lam_fn = make_ramp(li, lf, t_start=0.05, duration=RAMP_FAST)

    _, _, h1, _ = simulate_lambda(lam_fn, T=T_SIM)
    _, _, h2, _ = simulate_hill(lam_fn, T=T_SIM)
    _, _, h3, _ = simulate_kmhm(lam_fn, q_target, T=T_SIM, ramp_dur=RAMP_FAST)

    ep_results['R-C Ca²⁺ λ'].append(h1[-1].copy())
    ep_results['R-C Hill'].append(h2[-1].copy())
    ep_results['KMHM'].append(h3[-1].copy())
    print(f"C = {C:.2f} done")
# ════════════════════════════════════════════════════════════════

### Part B: Endpoint shift vs cocontraction

Plot the final x-position as a function of C for each model. If the C-command
only changes stiffness (Gribble's claim), the endpoints should be identical
across all C values.

In [ ]:
# ── Figure 3: Endpoint shift vs C ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

markers = {'R-C Ca²⁺ λ': 'o', 'R-C Hill': 's', 'KMHM': '^'}
lstyles = {'R-C Ca²⁺ λ': '-', 'R-C Hill': '-', 'KMHM': '--'}
colors  = {'R-C Ca²⁺ λ': C_CA, 'R-C Hill': C_HILL, 'KMHM': C_KMHM}
msizes  = {'R-C Ca²⁺ λ': 10, 'R-C Hill': 9, 'KMHM': 10}

ax = axes[0]; ax.set_title('A: Final x-position vs cocontraction', fontweight='bold')
for name in ['R-C Ca²⁺ λ', 'R-C Hill', 'KMHM']:
    xs = [ep[0]*100 for ep in ep_results[name]]
    ax.plot(C_values, xs, marker=markers[name], color=colors[name], lw=2,
            ms=msizes[name], ls=lstyles[name], label=name, zorder=3 if name=='R-C Hill' else 2)
ax.axhline(target_xy[0]*100, color='gray', ls=':', lw=1, label='Target x')
ax.set_xlabel('Cocontraction C (rad)'); ax.set_ylabel('Final x (cm)')
ax.legend(fontsize=9)
ax.annotate('R-C Hill and KMHM overlap\n(same static equilibrium)',
            xy=(0.50, 7.5), fontsize=8, style='italic', color='gray')

ax = axes[1]; ax.set_title('B: Endpoint error vs cocontraction', fontweight='bold')
for name in ['R-C Ca²⁺ λ', 'R-C Hill', 'KMHM']:
    errs = [np.linalg.norm(ep - target_xy)*100 for ep in ep_results[name]]
    ax.plot(C_values, errs, marker=markers[name], color=colors[name], lw=2,
            ms=msizes[name], ls=lstyles[name], label=name, zorder=3 if name=='R-C Hill' else 2)
ax.set_xlabel('Cocontraction C (rad)'); ax.set_ylabel('Endpoint error (cm)')
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

### Part C: Perturbation recovery at low vs high C

Does increasing cocontraction help the Hill models damp their oscillations?
Compare perturbation recovery at C = 0.10 (low stiffness) and C = 0.50 (high stiffness).

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Define perturbation function, then simulate R-C Hill
#   at C=0.10 and C=0.50 (both perturbed and unperturbed)
# ════════════════════════════════════════════════════════════════
def pert_fn(t):
    ...  # YOUR CODE HERE

pert_data = {}
for C in [0.10, 0.50]:
    li = lambda_for_posture(Q_REF, C=C)
    lf = lambda_for_posture(q_target, C=C)
    lam_fn = make_ramp(li, lf, t_start=0.05, duration=RAMP_FAST)
    t, _, h_u, _ = simulate_hill(lam_fn, T=T_SIM)
    ...  # YOUR CODE HERE
    pert_data[C] = (t, h_u, h_p)

print("Perturbation simulations complete.")
# ════════════════════════════════════════════════════════════════

In [ ]:
# ── Figure 4: Perturbation recovery at low vs high C ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for idx, C in enumerate([0.10, 0.50]):
    ax = axes[idx]
    t, h_u, h_p = pert_data[C]
    diff = np.linalg.norm(h_u - h_p, axis=1) * 100

    ax.set_title(f'R-C Hill perturbation recovery (C = {C})', fontweight='bold')
    ax.plot(t*1000, np.linalg.norm(h_u - start_hand, axis=1)*100,
            color=C_HILL, lw=2, label='Unperturbed')
    ax.plot(t*1000, np.linalg.norm(h_p - start_hand, axis=1)*100,
            color=C_HILL, lw=2, ls='--', alpha=0.6, label='Perturbed')
    ax.axhline(12, color='gray', ls=':', lw=1)
    ax.axvline(300, color='gray', ls=':', lw=0.8)
    ax.set_xlabel('Time (ms)'); ax.set_ylabel('Displacement (cm)')
    ax.legend(fontsize=8); ax.set_xlim(0, 2000)

plt.tight_layout(); plt.show()

# Print endpoint differences
for C in [0.10, 0.50]:
    t, h_u, h_p = pert_data[C]
    ep_diff = np.linalg.norm(h_u[-1] - h_p[-1]) * 100
    print(f"C = {C}: endpoint diff = {ep_diff:.4f} cm")

### Part D: Questions

**Q1.** Does the final hand position change with C for the Gribble model? For the Hill models? Explain why, referring to the CE–SE force balance.

**Q2.** Does higher cocontraction reduce the oscillations in R-C Hill's perturbation recovery? Why or why not?

**Q3.** Kistemaker et al. (2006) reported up to 8° of static positioning error when a controller ignores tendon compliance. Convert your endpoint shift (in cm) to approximate joint angle error. Is the magnitude consistent with their finding?

*Write 2–3 sentences per question.*

In [ ]:
# ════════════════════════════════════════════════════════════════
# ▶ FILL IN: Your answers
# ════════════════════════════════════════════════════════════════

# Q1:
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# λ values symmetrically, so the equilibrium position is unchanged.
# In the Hill models, increasing C increases total muscle force,
# which stretches the tendons, shortening the CEs and shifting the
# equilibrium.

# Q2:
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# because the spring-mass dynamics scale with both stiffness and
# inertia. The oscillation frequency may increase but the number
# of cycles before settling may not decrease proportionally.

# Q3:
...  # YOUR CODE HERE
...  # YOUR CODE HERE
# our cocontraction range (0.10–0.75) may not span the same range
# they tested, and our muscle model parameters differ.
# ════════════════════════════════════════════════════════════════